In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Subset
from torchvision import transforms

import medicalDataLoader_Enes as medicalDataLoader
from utils_Enes import to_var, getTargetSegmentation, dice_per_class, hausdorff_per_class, keep_largest_component, show_confusion, show_predictions
from UNet_Base_Enes import UNet

DATA_DIR = '../../Data/'
RUNS = {
    '1A supervisé': '../baseline/models/final_dice',
    'Pseudo-labels': 'models/pseudo_labels',
    'Mean Teacher': 'models/mean_teacher',
}
CLASSES = ['VD', 'Myo', 'VG']

histories = {name: np.load(d + '/history.npy', allow_pickle=True).item() for name, d in RUNS.items()}

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for name, h in histories.items():
    dice = np.array(h['val_dice'], dtype=float).mean(axis=1)
    hd = np.nanmean(np.array(h['val_hd'], dtype=float), axis=1)
    axes[0].plot(dice, label='{} ({:.3f})'.format(name, dice[-10:].mean()))
    axes[1].plot(hd, label=name)
axes[0].set_ylim(0.6, 0.9)
axes[0].set_title('Dice moyen de validation')
axes[1].set_ylim(0, 25)
axes[1].set_title('Hausdorff moyen de validation (px)')
for ax in axes:
    ax.set_xlabel('Époque')
    ax.grid(alpha=0.3)
axes[0].legend(title='Moyenne des 10 dernières époques', fontsize=9)
axes[1].legend(fontsize=9)
plt.tight_layout()
plt.savefig('figures/courbes_validation.png', dpi=150)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for k, c in enumerate(CLASSES):
    for name, h in histories.items():
        axes[k].plot(np.array(h['val_dice'], dtype=float)[:, k], label=name)
    axes[k].set_ylim(0.5, 1)
    axes[k].set_title('Dice de validation : ' + c)
    axes[k].set_xlabel('Époque')
    axes[k].grid(alpha=0.3)
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.savefig('figures/dice_par_classe.png', dpi=150)
plt.show()

In [ ]:
h = histories['Mean Teacher']
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(np.array(h['val_dice'], dtype=float).mean(axis=1), label='Professeur (EMA)')
axes[0].plot(np.array(h['student_dice'], dtype=float).mean(axis=1), label='Élève')
axes[0].set_ylim(0, 0.9)
axes[0].set_title('Mean Teacher : professeur et élève (validation)')
axes[0].legend()
axes[1].plot(h['train_loss'], label='Loss supervisée (CE + Dice)')
axes[1].plot(h['cons_loss'], label='Loss de cohérence (MSE)')
axes[1].set_yscale('log')
axes[1].set_title('Mean Teacher : losses d\'entraînement')
axes[1].legend()
for ax in axes:
    ax.set_xlabel('Époque')
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('figures/mean_teacher.png', dpi=150)
plt.show()

In [ ]:
transform = transforms.Compose([transforms.ToTensor()])
val_set = medicalDataLoader.MedicalImageDataset('val', DATA_DIR, transform=transform, mask_transform=transform)
val_loader = DataLoader(val_set, batch_size=8, shuffle=False)


def load_model(path):
    net = UNet(num_classes=4)
    net.load_state_dict(torch.load(path, map_location='cpu'))
    if torch.cuda.is_available():
        net.cuda()
    net.eval()
    return net


@torch.no_grad()
def evaluate(models, loader):
    preds, targets = [], []
    for images, labels, _ in loader:
        images = to_var(images)
        probs = torch.stack([torch.softmax(net(images), dim=1) for net in models]).mean(dim=0)
        preds.append(probs.argmax(dim=1).cpu())
        targets.append(getTargetSegmentation(labels))
    preds = keep_largest_component(torch.cat(preds))
    targets = torch.cat(targets)
    return dice_per_class(preds, targets), hausdorff_per_class(preds, targets)

In [ ]:
rows = [
    ('1A supervisé', '../baseline/models/final_dice/best.pth'),
    ('Pseudo-labels', 'models/pseudo_labels/best.pth'),
    ('Mean Teacher', 'models/mean_teacher/last.pth'),
]

print('{:<26}{:>10} | {:>6}{:>6}{:>6}{:>8} | {:>6}{:>6}{:>6}'.format(
    'Méthode', '10 dern.', 'VD', 'Myo', 'VG', 'Moy.', 'HD VD', 'Myo', 'VG'))
for name, path in rows:
    last10 = np.array(histories[name]['val_dice'], dtype=float)[-10:].mean()
    dsc, hd = evaluate([load_model(path)], val_loader)
    print('{:<26}{:>10.3f} | {:>6.3f}{:>6.3f}{:>6.3f}{:>8.3f} | {:>6.1f}{:>6.1f}{:>6.1f}'.format(
        name, last10, *dsc, np.mean(dsc), *hd))

In [ ]:
net_1a = load_model('../baseline/models/final_dice/best.pth')
net_mt = load_model('models/mean_teacher/last.pth')
show_confusion(net_1a, val_loader, title='(1A supervisé)', post_process=True)
show_confusion(net_mt, val_loader, title='(Mean Teacher)', post_process=True)

In [ ]:
examples = Subset(val_set, [5, 20, 40, 60])
examples_loader = DataLoader(examples, batch_size=4, shuffle=False)
show_predictions(net_1a, examples_loader, n=4, title='1A supervisé', post_process=True)
show_predictions(net_mt, examples_loader, n=4, title='Mean Teacher', post_process=True)